### 구글 드라이브 연동

In [636]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0. 데이터 불러오기 및 확인

In [637]:
import pandas as pd
onlinesales = pd.read_csv('/content/drive/MyDrive/데이터리안/Data/onlinesales.csv')
customer = pd.read_csv('/content/drive/MyDrive/데이터리안/Data/customer.csv')

In [638]:
onlinesales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52924 entries, 0 to 52923
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    52924 non-null  object 
 1   order_id       52924 non-null  object 
 2   date           52924 non-null  object 
 3   product_id     52924 non-null  object 
 4   category       52924 non-null  object 
 5   quantity       52924 non-null  int64  
 6   avg_cost       52924 non-null  float64
 7   shipping_fee   52924 non-null  float64
 8   coupon_status  52924 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 3.6+ MB


In [639]:
onlinesales.describe()

,quantity,avg_cost,shipping_fee
count,52924.000000,52924.000000,52924.000000
mean,4.497638,52.237646,10.517630
std,20.104711,64.006882,19.475613
min,1.000000,0.390000,0.000000
25%,1.000000,5.700000,6.000000
50%,1.000000,16.990000,6.000000
75%,2.000000,102.130000,6.500000
max,900.000000,355.740000,521.360000


In [640]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1468 entries, 0 to 1467
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  1468 non-null   object
 1   sex          1468 non-null   object
 2   local        1468 non-null   object
 3   period       1468 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 46.0+ KB


In [641]:
customer.describe()

,period
count,1468.000000
mean,25.912125
std,13.959667
min,2.000000
25%,14.000000
50%,26.000000
75%,38.000000
max,50.000000


### 1. EDA

In [642]:
# onlinesales 테이블 데이터 10개 확인
onlinesales.head(10)

,customer_id,order_id,date,product_id,category,quantity,avg_cost,shipping_fee,coupon_status
0,USER_1358,Transaction_0000,2019-01-01,Product_0981,Nest-USA,1,153.71,6.5,Used
1,USER_1358,Transaction_0001,2019-01-01,Product_0981,Nest-USA,1,153.71,6.5,Used
2,USER_1358,Transaction_0002,2019-01-01,Product_0904,Office,1,2.05,6.5,Used
3,USER_1358,Transaction_0003,2019-01-01,Product_0203,Apparel,5,17.53,6.5,Not Used
4,USER_1358,Transaction_0003,2019-01-01,Product_0848,Bags,1,16.50,6.5,Used
5,USER_1358,Transaction_0003,2019-01-01,Product_0854,Bags,15,5.15,6.5,Used
6,USER_1358,Transaction_0003,2019-01-01,Product_0880,Drinkware,15,3.08,6.5,Not Used
7,USER_1358,Transaction_0003,2019-01-01,Product_0885,Drinkware,15,10.31,6.5,Clicked
8,USER_1358,Transaction_0003,2019-01-01,Product_0898,Drinkware,5,9.27,6.5,Used
9,USER_0190,Transaction_0003,2019-01-01,Product_0914,Office,52,0.98,6.5,Used


In [643]:
# customer 테이블 데이터 10개 확인
customer.head(10)

,customer_id,sex,local,period
0,USER_1358,male,Chicago,12
1,USER_0190,male,California,43
2,USER_0066,male,Chicago,33
3,USER_0345,female,California,30
4,USER_0683,male,California,49
5,USER_0730,male,California,32
6,USER_0585,female,New York,46
7,USER_1347,female,New Jersey,24
8,USER_0736,female,Chicago,40
9,USER_0541,male,California,43


In [644]:
# onlinesales 테이블의 결측치 확인해보기
onlinesales.isnull().sum()

,0
customer_id,0
order_id,0
date,0
product_id,0
category,0
quantity,0
avg_cost,0
shipping_fee,0
coupon_status,0


In [645]:
# onlinesales 테이블의 데이터 수집 기간 확인
onlinesales['date'].agg(['min', 'max'])

,date
min,2019-01-01
max,2019-12-31


In [646]:
# onlinesales 테이블의 총 매출, 평균 매출 확인
onlinesales['sales'] = onlinesales['quantity'] * onlinesales['avg_cost']
onlinesales['sales'].agg(['sum', 'mean']).astype(int)

,sales
sum,4670794
mean,88


In [647]:
# onlinesales 테이블의 총 배송비, 평균 배송비 확인
onlinesales['shipping_fee'].agg(['sum', 'mean']).astype(int)

,shipping_fee
sum,556635
mean,10


In [648]:
# onlinesales 테이블의 쿠폰 상태 컬럼에 무슨 값이 있는지
onlinesales['coupon_status'].unique()

array(['Used', 'Not Used', 'Clicked'], dtype=object)

In [649]:
# onlinesales 테이블의 카테고리 컬럼에 무슨 값이 있는지
onlinesales['category'].unique()

array(['Nest-USA', 'Office', 'Apparel', 'Bags', 'Drinkware', 'Lifestyle',
       'Notebooks & Journals', 'Headgear', 'Waze', 'Fun', 'Nest-Canada',
       'Backpacks', 'Google', 'Bottles', 'Gift Cards', 'More Bags',
       'Housewares', 'Android', 'Accessories', 'Nest'], dtype=object)

### 2. RFM 분석

In [650]:
# onlinesales 테이블의 고객별 Recency 계산
customer_status = onlinesales.groupby('customer_id')['date'].agg('max').reset_index() # 요약 데이터프레임 활용
customer_status = customer_status.rename(columns={'date':'last_order_date'}) # 컬럼명 변경
customer_status.head()

,customer_id,last_order_date
0,USER_0000,2019-09-15
1,USER_0001,2019-11-02
2,USER_0002,2019-10-19
3,USER_0003,2019-12-14
4,USER_0004,2019-09-15


In [651]:
# 19년 12월 주문까지는 'recent', 그 이전은 'past'
import numpy as np
rfm = customer_status.copy()
rfm['Recency'] = np.where(customer_status['last_order_date'] >= '2019-12-01', 'recent', 'past')
rfm.head()

,customer_id,last_order_date,Recency
0,USER_0000,2019-09-15,past
1,USER_0001,2019-11-02,past
2,USER_0002,2019-10-19,past
3,USER_0003,2019-12-14,recent
4,USER_0004,2019-09-15,past


In [652]:
# onlinesales 테이블의 고객별 Frequency 계산
customer_status['cnts_order'] = onlinesales.groupby('customer_id')['order_id'].agg('nunique').reset_index()['order_id']
customer_status.head()

,customer_id,last_order_date,cnts_order
0,USER_0000,2019-09-15,1
1,USER_0001,2019-11-02,31
2,USER_0002,2019-10-19,8
3,USER_0003,2019-12-14,11
4,USER_0004,2019-09-15,13


In [653]:
# 주문횟수가 30번 이상이면 'high', 아니면 'low'
rfm['cnts_order'] = customer_status['cnts_order']
rfm['Frequency'] = np.where(customer_status['cnts_order'] >= 30, 'high', 'low')
rfm.head()

,customer_id,last_order_date,Recency,cnts_order,Frequency
0,USER_0000,2019-09-15,past,1,low
1,USER_0001,2019-11-02,past,31,high
2,USER_0002,2019-10-19,past,8,low
3,USER_0003,2019-12-14,recent,11,low
4,USER_0004,2019-09-15,past,13,low


In [654]:
# onlinesales 테이블의 고객별 Monetary 계산
onlinesales['sales'] = onlinesales['quantity'] * onlinesales['avg_cost']
customer_status['sum_sales'] = onlinesales.groupby('customer_id')['sales'].agg('sum').reset_index()['sales']
customer_status.head()

,customer_id,last_order_date,cnts_order,sum_sales
0,USER_0000,2019-09-15,1,30.99
1,USER_0001,2019-11-02,31,13834.90
2,USER_0002,2019-10-19,8,1442.12
3,USER_0003,2019-12-14,11,1360.07
4,USER_0004,2019-09-15,13,1442.47


In [655]:
# 총 비용이 $1500 이상이면 'high', 아니면 'low'
rfm['sum_sales'] = customer_status['sum_sales']
rfm['Monetary'] = np.where(customer_status['sum_sales'] >= 1500, 'high', 'low')
rfm.head()

,customer_id,last_order_date,Recency,cnts_order,Frequency,sum_sales,Monetary
0,USER_0000,2019-09-15,past,1,low,30.99,low
1,USER_0001,2019-11-02,past,31,high,13834.90,high
2,USER_0002,2019-10-19,past,8,low,1442.12,low
3,USER_0003,2019-12-14,recent,11,low,1360.07,low
4,USER_0004,2019-09-15,past,13,low,1442.47,low


In [656]:
# Recency별 고객 수 확인
rfm.groupby('Recency')['customer_id'].count()

,customer_id
Recency,
past,1232
recent,236


In [657]:
# Frequency별 고객 수 확인
rfm.groupby('Frequency')['customer_id'].count()

,customer_id
Frequency,
high,266
low,1202


In [658]:
# Monetary별 고객 수 확인
rfm.groupby('Monetary')['customer_id'].count()

,customer_id
Monetary,
high,798
low,670


In [682]:
# RFM별 고객 수 확인
rfm_new = customer_status[['customer_id']].copy()

rfm_new['Recency'] = np.where(customer_status['last_order_date'] >= '2019-12-01', 'recent', 'past')
rfm_new['Frequency'] = np.where(customer_status['cnts_order'] >= 30, 'high', 'low')
rfm_new['Monetary'] = np.where(customer_status['sum_sales'] >= 1500, 'high', 'low')

rfm_new = rfm_new.groupby(['Recency', 'Frequency', 'Monetary']).size().reset_index(name='customers')

rfm_new

,Recency,Frequency,Monetary,customers
0,past,high,high,206
1,past,low,high,436
2,past,low,low,590
3,recent,high,high,60
4,recent,low,high,96


### 3. 피봇 테이블

In [660]:
# date 컬럼 날짜 형식으로 변환
onlinesales['date'] = pd.to_datetime(onlinesales['date'])

In [661]:
# 'quarter' 열 추가 (분기 구분)
onlinesales['quarter'] = pd.cut(
    onlinesales['date'].dt.month,
    bins=[0, 3, 6, 9, 12],
    labels=['Q1', 'Q2', 'Q3', 'Q4'],
    right=True
)
onlinesales['quarter']

,quarter
0,Q1
1,Q1
2,Q1
3,Q1
4,Q1
...,...
52919,Q4
52920,Q4
52921,Q4
52922,Q4


In [662]:
pivot_table = pd.pivot_table(
    onlinesales,
    index='coupon_status',             # 행 인덱스
    columns='quarter',       # 열 인덱스
    values='order_id',          # 집계할 값
    aggfunc='nunique',           # 집계 함수
    fill_value=0             # 결측값을 0으로 채우기
)
pivot_table

quarter,Q1,Q2,Q3,Q4
coupon_status,,,,
Clicked,3688,3877,4364,4442
Not Used,1492,1581,1839,1653
Used,2735,2921,3369,3233


In [663]:
# 쿠폰 상태별 총 주문 수 추가
pivot_table['Total'] = pivot_table.sum(axis=1)

pivot_table

### 4. 데이터 연결하기

- 위아래로 연결하기

In [665]:
titanic = pd.read_csv('/content/drive/MyDrive/데이터리안/Data/train.csv')
titanic_1 = titanic.copy()

In [666]:
# 위아래로 연결하기
combined_df = pd.concat([titanic, titanic_1], axis = 0, ignore_index = True)
combined_df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [667]:
combined_df.loc[888:893]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.7500,NaN,Q
891,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
892,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
893,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [668]:
# 단순하게 좌우로 합치기
combined_df1 = pd.concat([titanic, titanic_1], axis = 1, ignore_index = False)
combined_df1.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,...,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,...,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,...,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,...,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


- INNER JOIN

In [669]:
onlinesales = pd.read_csv('/content/drive/MyDrive/데이터리안/Data/onlinesales.csv')
customer = pd.read_csv('/content/drive/MyDrive/데이터리안/Data/customer.csv')

In [670]:
# customer 테이블과 onlinesales 테이블을 INNER JOIN
# merge 함수 사용
df = pd.merge(customer, onlinesales, on = 'customer_id', how = 'inner')
df.head()

,customer_id,sex,local,period,order_id,date,product_id,category,quantity,avg_cost,shipping_fee,coupon_status
0,USER_1358,male,Chicago,12,Transaction_0000,2019-01-01,Product_0981,Nest-USA,1,153.71,6.5,Used
1,USER_1358,male,Chicago,12,Transaction_0001,2019-01-01,Product_0981,Nest-USA,1,153.71,6.5,Used
2,USER_1358,male,Chicago,12,Transaction_0002,2019-01-01,Product_0904,Office,1,2.05,6.5,Used
3,USER_1358,male,Chicago,12,Transaction_0003,2019-01-01,Product_0203,Apparel,5,17.53,6.5,Not Used
4,USER_1358,male,Chicago,12,Transaction_0003,2019-01-01,Product_0848,Bags,1,16.50,6.5,Used


In [671]:
# 성별이 여성인 데이터만 추출
df_female = df[df['sex'] == 'female']
df_female.head()

,customer_id,sex,local,period,order_id,date,product_id,category,quantity,avg_cost,shipping_fee,coupon_status
383,USER_0345,female,California,30,Transaction_0013,2019-01-01,Product_0971,Lifestyle,1,1.24,6.5,Clicked
425,USER_0585,female,New York,46,Transaction_0038,2019-01-01,Product_0981,Nest-USA,1,153.71,6.5,Clicked
426,USER_0585,female,New York,46,Transaction_0039,2019-01-01,Product_0976,Nest-USA,1,122.77,6.5,Clicked
427,USER_0585,female,New York,46,Transaction_0040,2019-01-01,Product_0981,Nest-USA,1,153.71,6.5,Used
428,USER_0585,female,New York,46,Transaction_0041,2019-01-02,Product_0925,Headgear,2,19.59,6.5,Clicked


In [672]:
# 성별이 여성인 고객의 고객id, 성별, 지역, 제품id, 제품분류, 쿠폰 상태, 할인율 추출
discount = pd.read_csv('/content/drive/MyDrive/데이터리안/Data/discount.csv')

merged_df = pd.merge(df_female, discount, on = 'category', how = 'inner')

merged_df.head()

,customer_id,sex,local,period,order_id,date,product_id,category,quantity,avg_cost,shipping_fee,coupon_status,month,coupon_code,discount_rate
0,USER_0345,female,California,30,Transaction_0013,2019-01-01,Product_0971,Lifestyle,1,1.24,6.5,Clicked,Jan,EXTRA10,10
1,USER_0345,female,California,30,Transaction_0013,2019-01-01,Product_0971,Lifestyle,1,1.24,6.5,Clicked,Feb,EXTRA20,20
2,USER_0345,female,California,30,Transaction_0013,2019-01-01,Product_0971,Lifestyle,1,1.24,6.5,Clicked,Mar,EXTRA30,30
3,USER_0345,female,California,30,Transaction_0013,2019-01-01,Product_0971,Lifestyle,1,1.24,6.5,Clicked,Apr,EXTRA10,10
4,USER_0345,female,California,30,Transaction_0013,2019-01-01,Product_0971,Lifestyle,1,1.24,6.5,Clicked,May,EXTRA20,20


In [673]:
merged_df = merged_df[['customer_id', 'sex', 'local', 'product_id', 'category', 'coupon_status', 'discount_rate']].sort_values('customer_id').reset_index(drop = True)
merged_df.head()

,customer_id,sex,local,product_id,category,coupon_status,discount_rate
0,USER_0000,female,New York,Product_0048,Apparel,Used,10
1,USER_0000,female,New York,Product_0946,Office,Used,10
2,USER_0000,female,New York,Product_0946,Office,Used,20
3,USER_0000,female,New York,Product_0946,Office,Used,30
4,USER_0000,female,New York,Product_0946,Office,Used,10


In [674]:
onlinesales = pd.read_csv('/content/drive/MyDrive/데이터리안/Data/onlinesales.csv')
customer = pd.read_csv('/content/drive/MyDrive/데이터리안/Data/customer.csv')

In [675]:
# join 함수 사용
# index를 기준으로 결합하여 join 전에 가공이 필요
customer = customer.set_index(keys = 'customer_id')
onlinesales = onlinesales.set_index(keys = 'customer_id')

df1 = customer.join(onlinesales, how = 'inner')
df1.head()

,sex,local,period,order_id,date,product_id,category,quantity,avg_cost,shipping_fee,coupon_status
customer_id,,,,,,,,,,,
USER_0000,female,New York,31,Transaction_16900,2019-09-15,Product_0048,Apparel,1,19.99,75.00,Used
USER_0000,female,New York,31,Transaction_16900,2019-09-15,Product_0946,Office,2,5.50,75.00,Used
USER_0001,male,New York,20,Transaction_5262,2019-03-24,Product_0945,Office,1,2.99,14.41,Clicked
USER_0001,male,New York,20,Transaction_5262,2019-03-24,Product_0965,Office,1,9.99,14.41,Used
USER_0001,male,New York,20,Transaction_5263,2019-03-24,Product_0981,Nest-USA,1,149.00,6.50,Used


In [676]:
# 성별이 여성인 데이터만 추출
df1_female = df1[df1['sex'] == 'female']
df1_female.head()

,sex,local,period,order_id,date,product_id,category,quantity,avg_cost,shipping_fee,coupon_status
customer_id,,,,,,,,,,,
USER_0000,female,New York,31,Transaction_16900,2019-09-15,Product_0048,Apparel,1,19.99,75.0,Used
USER_0000,female,New York,31,Transaction_16900,2019-09-15,Product_0946,Office,2,5.50,75.0,Used
USER_0004,female,Chicago,31,Transaction_16887,2019-09-15,Product_1046,Apparel,1,11.89,6.0,Used
USER_0004,female,Chicago,31,Transaction_16888,2019-09-15,Product_0900,Drinkware,1,10.39,6.5,Clicked
USER_0004,female,Chicago,31,Transaction_16889,2019-09-15,Product_0352,Apparel,1,28.00,6.0,Clicked


In [677]:
# 성별이 여성인 고객의 고객id, 성별, 지역, 제품id, 제품분류, 쿠폰 상태, 할인율 추출
df1_female = df1_female.reset_index()
df1_female = df1_female.set_index(keys = 'category')
df1_female.head()

,customer_id,sex,local,period,order_id,date,product_id,quantity,avg_cost,shipping_fee,coupon_status
category,,,,,,,,,,,
Apparel,USER_0000,female,New York,31,Transaction_16900,2019-09-15,Product_0048,1,19.99,75.0,Used
Office,USER_0000,female,New York,31,Transaction_16900,2019-09-15,Product_0946,2,5.50,75.0,Used
Apparel,USER_0004,female,Chicago,31,Transaction_16887,2019-09-15,Product_1046,1,11.89,6.0,Used
Drinkware,USER_0004,female,Chicago,31,Transaction_16888,2019-09-15,Product_0900,1,10.39,6.5,Clicked
Apparel,USER_0004,female,Chicago,31,Transaction_16889,2019-09-15,Product_0352,1,28.00,6.0,Clicked


In [678]:
discount = discount.set_index(keys = 'category')
discount.head()

,month,coupon_code,discount_rate
category,,,
Apparel,Jan,SALE10,10
Apparel,Feb,SALE20,20
Apparel,Mar,SALE30,30
Nest-USA,Jan,ELEC10,10
Nest-USA,Feb,ELEC20,20


In [679]:
joined_df = df1_female.join(discount, how = 'inner')
joined_df.head()

,customer_id,sex,local,period,order_id,date,product_id,quantity,avg_cost,shipping_fee,coupon_status,month,coupon_code,discount_rate
category,,,,,,,,,,,,,,
Accessories,USER_0032,female,California,9,Transaction_20738,2019-11-10,Product_0182,1,2.39,12.91,Clicked,Jan,ACC10,10
Accessories,USER_0032,female,California,9,Transaction_20738,2019-11-10,Product_0182,1,2.39,12.91,Clicked,Feb,ACC20,20
Accessories,USER_0032,female,California,9,Transaction_20738,2019-11-10,Product_0182,1,2.39,12.91,Clicked,Mar,ACC30,30
Accessories,USER_0032,female,California,9,Transaction_20738,2019-11-10,Product_0182,1,2.39,12.91,Clicked,Apr,ACC10,10
Accessories,USER_0032,female,California,9,Transaction_20738,2019-11-10,Product_0182,1,2.39,12.91,Clicked,May,ACC20,20


In [680]:
joined_df = joined_df.reset_index()
joined_df = joined_df[['customer_id', 'sex', 'local', 'product_id', 'category', 'coupon_status', 'discount_rate']].sort_values('customer_id').reset_index(drop = True)
joined_df.head()

,customer_id,sex,local,product_id,category,coupon_status,discount_rate
0,USER_0000,female,New York,Product_0946,Office,Used,10
1,USER_0000,female,New York,Product_0048,Apparel,Used,10
2,USER_0000,female,New York,Product_0048,Apparel,Used,20
3,USER_0000,female,New York,Product_0048,Apparel,Used,30
4,USER_0000,female,New York,Product_0048,Apparel,Used,10
